In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/march-machine-learning-mania-2026/Conferences.csv
/kaggle/input/march-machine-learning-mania-2026/WNCAATourneyDetailedResults.csv
/kaggle/input/march-machine-learning-mania-2026/WRegularSeasonCompactResults.csv
/kaggle/input/march-machine-learning-mania-2026/MNCAATourneySeedRoundSlots.csv
/kaggle/input/march-machine-learning-mania-2026/MRegularSeasonDetailedResults.csv
/kaggle/input/march-machine-learning-mania-2026/MNCAATourneyCompactResults.csv
/kaggle/input/march-machine-learning-mania-2026/MGameCities.csv
/kaggle/input/march-machine-learning-mania-2026/WSecondaryTourneyCompactResults.csv
/kaggle/input/march-machine-learning-mania-2026/WGameCities.csv
/kaggle/input/march-machine-learning-mania-2026/MSeasons.csv
/kaggle/input/march-machine-learning-mania-2026/WNCAATourneySlots.csv
/kaggle/input/march-machine-learning-mania-2026/MSecondaryTourneyTeams.csv
/kaggle/input/march-machine-learning-mania-2026/SampleSubmissionStage2.csv
/kaggle/input/march-machine-learning-mania

In [2]:
import os
import warnings
from itertools import combinations
from scipy.special import expit  
warnings.filterwarnings("ignore")

In [3]:
data_dir       = "/kaggle/input/march-machine-learning-mania-2026"
sub_file_path  = f"{data_dir}/SampleSubmissionStage1.csv"

In [4]:
M_regular_results = pd.read_csv(f"{data_dir}/MRegularSeasonDetailedResults.csv")
M_tourney_results = pd.read_csv(f"{data_dir}/MNCAATourneyDetailedResults.csv")
M_seeds = pd.read_csv(f"{data_dir}/MNCAATourneySeeds.csv")

W_regular_results = pd.read_csv(f"{data_dir}/WRegularSeasonDetailedResults.csv")
W_tourney_results = pd.read_csv(f"{data_dir}/WNCAATourneyDetailedResults.csv")
W_seeds = pd.read_csv(f"{data_dir}/WNCAATourneySeeds.csv")

In [5]:
M_regular_results.nunique()

Season      24
DayNum     133
WTeamID    372
WScore     104
LTeamID    372
LScore     105
WLoc         3
NumOT        7
WFGM        45
WFGA        73
WFGM3       27
WFGA3       57
WFTM        50
WFTA        66
WOR         37
WDR         47
WAst        39
WTO         34
WStl        27
WBlk        21
WPF         39
LFGM        42
LFGA        75
LFGM3       23
LFGA3       58
LFTM        44
LFTA        61
LOR         36
LDR         44
LAst        32
LTO         42
LStl        23
LBlk        19
LPF         42
dtype: int64

In [6]:
M_regular_results.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,...,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,...,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,...,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,...,16,17,27,21,15,12,10,7,1,14


In [7]:
SEASON   = 2026
ELO_INIT = 1500
ELO_K    = 32

def compute_advanced_stats(detailed_df):

    stats_list = []

    for _, row in detailed_df.iterrows():
        season = row["Season"]

        for prefix, opp_prefix in [("W", "L"), ("L", "W")]:
            tid     = row[f"{prefix}TeamID"]
            fgm     = row.get(f"{prefix}FGM",  0)
            fga     = row.get(f"{prefix}FGA",  1)
            fgm3    = row.get(f"{prefix}FGM3", 0)
            fga3    = row.get(f"{prefix}FGA3", 1)
            ftm     = row.get(f"{prefix}FTM",  0)
            fta     = row.get(f"{prefix}FTA",  0)
            orb     = row.get(f"{prefix}OR",   0)
            drb     = row.get(f"{prefix}DR",   0)
            ast     = row.get(f"{prefix}Ast",  0)
            to      = row.get(f"{prefix}TO",   1)
            stl     = row.get(f"{prefix}Stl",  0)
            blk     = row.get(f"{prefix}Blk",  0)
            score   = row.get(f"{prefix}Score",0)
            opp_sc  = row.get(f"{opp_prefix}Score", 0)

            # our metrics
            efg     = (fgm + 0.5 * fgm3) / max(fga, 1)        # Effective FG
            tov_r   = to / max(fga + 0.44 * fta + to, 1)       # Turnover Rate
            ast_r   = ast / max(fgm, 1)                         # Assist Ratio
            reb     = orb + drb                                  # Total Rebounds
            pt_diff = score - opp_sc                            # Point Differential

            stats_list.append({
                "Season" : season,
                "TeamID" : tid,
                "eFG"    : efg,
                "TovRate": tov_r,
                "AstRate": ast_r,
                "Rebounds": reb,
                "PtDiff" : pt_diff,
                "Score"  : score,
            })

    df = pd.DataFrame(stats_list)
    agg = df.groupby(["Season", "TeamID"]).agg(
        eFG      = ("eFG",     "mean"),
        TovRate  = ("TovRate", "mean"),
        AstRate  = ("AstRate", "mean"),
        Rebounds = ("Rebounds","mean"),
        PtDiff   = ("PtDiff",  "mean"),
        AvgScore = ("Score",   "mean"),
    ).reset_index()

    return agg

In [8]:
 #ELO RATING (tourney_games = 2x weight)
def build_elo(reg_df, tourney_df, k=ELO_K, init=ELO_INIT):
    elo        = {}
    season_elo = {}

    all_games = pd.concat([
        reg_df.assign(IsPlayoff=0),
        tourney_df.assign(IsPlayoff=1)
    ]).sort_values(["Season", "DayNum"]).reset_index(drop=True)

    for _, row in all_games.iterrows():
        w, l   = row["WTeamID"], row["LTeamID"]
        season = row["Season"]
        ew = elo.get(w, init)
        el = elo.get(l, init)

        exp_w  = 1 / (1 + 10 ** ((el - ew) / 400))
        eff_k  = k * (2 if row["IsPlayoff"] else 1)

        elo[w] = ew + eff_k * (1 - exp_w)
        elo[l] = el + eff_k * (0 - (1 - exp_w))

        if row["DayNum"] >= 132:
            season_elo[(season, w)] = elo[w]
            season_elo[(season, l)] = elo[l]

    return season_elo

In [9]:
#seed metric
def seed_to_int(s):
    return int("".join(filter(str.isdigit, str(s))))

def build_seed_strength(seeds_df, tourney_df):
    s = seeds_df.copy()
    s["SeedNum"] = s["Seed"].apply(seed_to_int)
    wins = tourney_df.groupby(["Season","WTeamID"]).size().reset_index(name="Wins")
    wins.rename(columns={"WTeamID":"TeamID"}, inplace=True)
    merged = s.merge(wins, on=["Season","TeamID"], how="left").fillna(0)
    return merged.groupby("SeedNum")["Wins"].mean()

In [10]:
# tourney win rate
def build_team_winrate(tourney_df):
    w = tourney_df.groupby("WTeamID").size().rename("Wins")
    l = tourney_df.groupby("LTeamID").size().rename("Losses")
    rec = pd.concat([w, l], axis=1).fillna(0)
    rec["WinRate"] = rec["Wins"] / (rec["Wins"] + rec["Losses"]).replace(0, np.nan)
    return rec["WinRate"].fillna(0.5)

In [11]:
#Tteam strength score (Elo + Seed + WinRate + calsulated Advanced Stats)

def get_strength(team_id, season, seeds_df, elo_dict,
                 seed_strength, team_wr, adv_stats):

    
    elo_val  = elo_dict.get((season, team_id),
               elo_dict.get((season-1, team_id), ELO_INIT))
    elo_norm = (elo_val - 1200) / 600

  
    sr = seeds_df[(seeds_df.Season==season) & (seeds_df.TeamID==team_id)]
    sn = seed_to_int(sr["Seed"].values[0]) if len(sr) > 0 else 8
    seed_score = seed_strength.get(sn, seed_strength.mean())
    seed_norm  = seed_score / (seed_strength.max() + 1e-9)

    
    wr = team_wr.get(team_id, 0.5)

    # 4. taking range DetailedResults — our extra edge
    adv = adv_stats[(adv_stats.Season==season) & (adv_stats.TeamID==team_id)]
    if len(adv) > 0:
        efg     = adv["eFG"].values[0]
        pt_diff = adv["PtDiff"].values[0] / 30.0   
        tov     = 1 - adv["TovRate"].values[0]      # lower tov is better
    else:
        efg, pt_diff, tov = 0.5, 0.0, 0.5

    # Blended score (here ! weights tuned for Brier Score minimization)
    # Elo=40%, Seed=25%, WinRate=15%, eFG=10%, PtDiff=7%, ToV=3%
    score = (0.40 * elo_norm +
             0.25 * seed_norm +
             0.15 * wr +
             0.10 * efg +
             0.07 * pt_diff +
             0.03 * tov)

    return score, elo_val



In [12]:
# sigmoid 
def win_probability(s1, e1, s2, e2):
    prob_score = expit((s1 - s2) * 10)              
    prob_elo   = 1 / (1 + 10 ** ((e2 - e1) / 400)) 
    final      = 0.6 * prob_score + 0.4 * prob_elo
    return float(np.clip(final, 0.025, 0.975))

In [13]:

#validation of  brier score 

def validate(tourney_df, seeds_df, elo_dict, seed_str, team_wr, adv_stats, label):
    print(f"\nValidating [{label}] on 2021–2025 tournament games...")
    scores = []
    for season in [2021, 2022, 2023, 2024, 2025]:
        games = tourney_df[tourney_df.Season == season]
        for _, row in games.iterrows():
            wt, lt = row["WTeamID"], row["LTeamID"]
            t1, t2 = (wt, lt) if wt < lt else (lt, wt)
            true_label = 1 if wt < lt else 0

            s1, e1 = get_strength(t1, season, seeds_df, elo_dict, seed_str, team_wr, adv_stats)
            s2, e2 = get_strength(t2, season, seeds_df, elo_dict, seed_str, team_wr, adv_stats)
            prob   = win_probability(s1, e1, s2, e2)
            scores.append((prob - true_label) ** 2)

    print(f"   Brier Score: {np.mean(scores):.5f}  (lower is better | baseline=0.25)")
    return np.mean(scores)

In [14]:
def build_strength_lookup(seeds_df, elo_dict, seed_str, team_wr, adv_stats, season):
    
    all_teams = seeds_df["TeamID"].unique()
    lookup    = {}

    for tid in all_teams:
        s, e = get_strength(tid, season, seeds_df, elo_dict, seed_str, team_wr, adv_stats)
        lookup[tid] = (s, e)

    return lookup

In [15]:
# i am using SampleSubmission IDs to create submission.csv
def generate_submission(sub_file, seeds_m, seeds_w,
                        elo_m, elo_w, seed_str_m, seed_str_w,
                        team_wr_m, team_wr_w,
                        adv_m, adv_w, season=SEASON):

    print(f"\nGenerating predictions (Season {season})...")
    import time
    t0 = time.time()

    sub = pd.read_csv(sub_file)


    split         = sub["ID"].str.split("_", expand=True)
    sub["T1"]     = split[1].astype(int)
    sub["T2"]     = split[2].astype(int)

    print(" Pre-computing team strengths...")
    lookup_m = build_strength_lookup(seeds_m, elo_m, seed_str_m, team_wr_m, adv_m, season)
    lookup_w = build_strength_lookup(seeds_w, elo_w, seed_str_w, team_wr_w, adv_w, season)

   
    def lookup_to_df(lookup):
        return pd.DataFrame(
            [(tid, s, e) for tid, (s, e) in lookup.items()],
            columns=["TeamID", "Strength", "Elo"]
        )

    df_m = lookup_to_df(lookup_m)
    df_w = lookup_to_df(lookup_w)
    df_all = pd.concat([df_m, df_w], ignore_index=True)  # one combined table

    print("Merging strength scores...")
    sub = sub.merge(df_all.rename(columns={"TeamID":"T1","Strength":"S1","Elo":"E1"}),
                    on="T1", how="left")
    sub = sub.merge(df_all.rename(columns={"TeamID":"T2","Strength":"S2","Elo":"E2"}),
                    on="T2", how="left")

    
    sub["S1"] = sub["S1"].fillna(0.0)
    sub["S2"] = sub["S2"].fillna(0.0)
    sub["E1"] = sub["E1"].fillna(ELO_INIT)
    sub["E2"] = sub["E2"].fillna(ELO_INIT)

    print(" Computing probabilities (vectorized)...")
    score_diff = sub["S1"].values - sub["S2"].values
    elo_diff   = sub["E2"].values - sub["E1"].values

    prob_score = 1 / (1 + np.exp(-score_diff * 10))     # sigmoid
    prob_elo   = 1 / (1 + 10 ** (elo_diff / 400))       

    sub["Pred"] = np.round(
        np.clip(0.60 * prob_score + 0.40 * prob_elo, 0.025, 0.975), 5
    )

    result = sub[["ID", "Pred"]]

    print(f" {len(result):,} predictions in {time.time()-t0:.2f} seconds!")
    return result


In [16]:
print("=" * 56)
print(" NCAA March Mania 2026 — Running Pipeline")
print("=" * 56)

print("\nComputing advanced stats from DetailedResults...")
adv_stats_m = compute_advanced_stats(M_regular_results)
adv_stats_w = compute_advanced_stats(W_regular_results)
print(f"   Men's  adv stats: {adv_stats_m.shape}")
print(f"   Women's adv stats: {adv_stats_w.shape}")

print("\nBuilding Elo ratings...")
elo_m = build_elo(M_regular_results, M_tourney_results)
elo_w = build_elo(W_regular_results, W_tourney_results)


print("Building seed strength matrices...")
seed_str_m = build_seed_strength(M_seeds, M_tourney_results)
seed_str_w = build_seed_strength(W_seeds, W_tourney_results)


print("Building team tournament win rates...")
team_wr_m = build_team_winrate(M_tourney_results)
team_wr_w = build_team_winrate(W_tourney_results)

validate(M_tourney_results, M_seeds, elo_m, seed_str_m, team_wr_m, adv_stats_m, "Men's")
validate(W_tourney_results, W_seeds, elo_w, seed_str_w, team_wr_w, adv_stats_w, "Women's")


submission = generate_submission(
    sub_file_path,
    M_seeds, W_seeds,
    elo_m, elo_w,
    seed_str_m, seed_str_w,
    team_wr_m, team_wr_w,
    adv_stats_m, adv_stats_w
)

submission.to_csv("submission.csv", index=False)
print(submission.head(10).to_string(index=False))



 NCAA March Mania 2026 — Running Pipeline

Computing advanced stats from DetailedResults...
   Men's  adv stats: (8346, 8)
   Women's adv stats: (5965, 8)

Building Elo ratings...
Building seed strength matrices...
Building team tournament win rates...

Validating [Men's] on 2021–2025 tournament games...
   Brier Score: 0.15014  (lower is better | baseline=0.25)

Validating [Women's] on 2021–2025 tournament games...
   Brier Score: 0.10304  (lower is better | baseline=0.25)

Generating predictions (Season 2026)...
 Pre-computing team strengths...
Merging strength scores...
 Computing probabilities (vectorized)...
 519,144 predictions in 2.08 seconds!
            ID    Pred
2022_1101_1102 0.60333
2022_1101_1103 0.20579
2022_1101_1104 0.02500
2022_1101_1105 0.56509
2022_1101_1106 0.83063
2022_1101_1107 0.53950
2022_1101_1108 0.50285
2022_1101_1110 0.70475
2022_1101_1111 0.53753
2022_1101_1112 0.02884
